# 119. 银行营销响应预测项目

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 34 / 34 步：把完整流程迁移到真实项目**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 共享单车需求预测项目  →  **本章任务：** 银行营销响应预测项目  →  **下一步：** 模块大作业《模型上线评审会》
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

**背景引入**：一家银行的客户经理每天要打出成百上千通电话，可绝大多数客户并不需要新的理财产品。与其广撒网打扰所有人，不如把有限的电话优先打给更可能认购的客户。这个项目要做的，就是利用客户画像和过往活动记录，提前判断一次营销电话值不值得打，并评估这样筛选能省下多少人力和精力。数据来自 UCI Bank Marketing 公开集，记录了客户基本信息和上一次活动的电话结果，目标字段 `y` 表示客户是否认购了定期存款。



## 本章目标

学完本章，你将能够：

- **理解**：理解「银行营销响应预测项目」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「银行营销响应预测项目」的关键输出指标。
- **迁移**：能把「银行营销响应预测项目」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 119.1 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| age/job/education | 客户画像 | 数值与类别特征 |
| duration | 本次通话时长 | 通话结束后才知道，禁止使用 |
| campaign/pdays/previous | 接触历史 | 活动相关特征 |
| poutcome | 上次活动结果 | 历史信号 |
| y | 是否认购 | 二分类目标 |

## 119.2 数据质量检查清单

- 分号分隔及字段类型
- unknown的数量和含义
- 重复记录与campaign长尾
- 正类比例和多数类准确率
- duration事后泄漏（打个比方：`duration`（通话时长）要等电话打完才知道，拿它预测“这次会不会成交”，等于先看到结局再倒推开场，是典型的“偷看未来”。）
- 三组数据的类别比例


## 119.3 项目任务

1. 明确通话前预测时点与目标
2. 审计数据质量和类别不平衡
3. 清理重复并探索响应差异
4. 识别duration等禁止字段
5. 分层划分训练、验证和测试
6. 建立混合类型预处理Pipeline
7. 比较Dummy、逻辑回归和随机森林
8. 评价PR-AUC、阈值、Lift与覆盖率
9. 分析错误类型和客户分组
10. 解释特征重要性并总结模型局限


## 119.4 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 1. 原始数据质量与类别不平衡审计 | `pd.read_csv()`、`raw.astype()`、`pd.Series()`、`raw.duplicated()` | 检查分隔符、重复、unknown、目标比例和长尾变量。 | 分号分隔及字段类型 |
| 2. 清理重复并探索响应差异 | `raw.drop_duplicates()`、`df.groupby()`、`target.agg()`、`df.target.mean()` | unknown保留为显式类别，因为未知并不等于否；描述性组间差异不代表营销因果效果。 | unknown的数量和含义 |
| 3. 定义预测时点并检查泄漏 | `df.groupby()`、`duration.mean()`、`.round()`、`set()` | duration只有通话结束后才能获得，在通话前响应预测中属于典型事后泄漏。 | 重复记录与campaign长尾 |
| 4. 分层划分与Dummy基线 | `dummy.predict_proba()`、`part.mean()`、`.fit()`、`df[features]` | 分层划分保持三组正类比例一致；验证集选模型，测试集只用于最终评价。 | 正类比例和多数类准确率 |
| 5. 建立混合类型预处理Pipeline | `X_train.select_dtypes()`、`columns.tolist()`、`cat[:6]`、`num[:6]` | 类别变量独热编码、数值变量标准化，并把预处理与模型绑定，避免数据处理泄漏。 | duration事后泄漏 |
| 6. 比较逻辑回归与随机森林 | `models.items()`、`model.fit()`、`rows.append()`、`model.predict_proba()` | 两个模型使用相同数据和预处理，在验证集PR-AUC上进行公平比较。 | 三组数据的类别比例 |
| 7. 测试集概率指标评价 | `pd.Series()`、`y_test.mean()`、`metrics.round()`、`.to_string()` | 类别不平衡时以PR-AUC为主，同时报告ROC-AUC和概率损失。 | 分号分隔及字段类型 |
| 8. 比较Top-K阈值、Lift与覆盖率 | `pd.DataFrame()`、`y_test.to_numpy()`、`ranked.head()`、`threshold_rows.append()` | Top-K用于教学性阈值评价，展示精确率、召回率和Lift之间的权衡。 | unknown的数量和含义 |
| 9. 错误类型与客户分组 | `np.select()`、`pd.cut()`、`error_df.groupby()`、`error_df.error_type.value_counts()` | 区分漏判响应与误报响应，并比较年龄段中的实际率、平均评分和错误率。 | 重复记录与campaign长尾 |
| 10. 特征解释与模型局限 | `np.linspace()`、`pd.Series()`、`importance.head()`、`.sort_values()` | 置换重要性说明模型主要利用哪些历史信号，同时强调响应预测不等于干预效果预测。 | 正类比例和多数类准确率 |


## 119.5 项目交付物

- 一份从数据审计到模型评价可完整运行的 Notebook
- 数据清洗前后样本变化和关键质量检查结果
- 基线与候选模型的指标对比表
- 错误切片、特征解释和有边界的业务结论

## 119.6 阶段检查点

- [ ] 数据与目标定义完成：样本粒度、预测时点和指标已写清楚
- [ ] 基线完成：知道复杂模型相对什么标准比较
- [ ] 模型评价完成：测试集只使用一次，并检查误差切片
- [ ] 交付完成：结论与证据对应，不把相关性写成因果

## 119.7 最低完成标准

- 每个代码阶段都有可见输出，不能依赖未展示的隐藏状态。
- 所有关键清洗、筛选和评价口径都写在 Markdown 或注释中。
- 最终结论至少引用一个数值或图表证据，并说明适用范围。

## 119.8 提升任务

完成基础验收后，可以增加一个对照方案、一个分组切片或一个参数敏感性实验，比较结果是否稳定。


## 119.9 原始数据质量与类别不平衡审计

检查分隔符、重复、unknown、目标比例和长尾变量。


<!-- math-foundation:chapter-119 -->
### 数学推导｜营销模型的期望净收益

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜阈值产生联系名单。** 联系人数 $A(t)=TP(t)+FP(t)$。

**第 2 步｜拆分收益与成本。** 成功响应带来价值 $TP(t)v$；无效联系产生额外代价 $FP(t)c$；每次行动还承担 $c_AA(t)$。

**第 3 步｜得到净收益并选阈值。** 

$$
EV(t)=TP(t)v-FP(t)c-c_AA(t),
\qquad
t^*=\arg\max_tEV(t)
$$

若名单有预算上限，还需同时满足 $A(t)\le A_{max}$，不能只追求无约束的最高收益。

**把上面的关系收束为本章计算式：**

$$
EV(t)=TP(t)v-FP(t)c-c_A\bigl(TP(t)+FP(t)\bigr)
$$

**符号解释：** $v$ 是一次成功转化价值，$c$ 是错误联系代价，$c_A$ 是行动成本。

**代码对应：** 基于验证集阈值表计算期望收益，并报告联系人数和风险。

**使用边界：** 价值与成本应来自可审计假设；历史收益不能保证未来分布不变。


In [ ]:
import numpy as np
import pandas as pd

raw = pd.read_csv("/datasets/bank_marketing_full.csv", sep=";")
unknown = (raw.astype(str) == "unknown").sum().sort_values(ascending=False)
positive_rate = (raw["y"] == "yes").mean()
majority_accuracy = max(positive_rate, 1 - positive_rate)
audit = pd.Series(
    {
        "行数": len(raw),
        "重复": raw.duplicated().sum(),
        "正类率": positive_rate,
        "多数类准确率": majority_accuracy,
        "campaign_P99": raw["campaign"].quantile(0.99),
    }
)
print(audit.round(3).to_string())
print("unknown 最多字段：\n", unknown.head(8))


**练一练**：刚才我们把原始记录读进了 `raw`。开工前得先看清这组数据有没有"坑"——除了 `y` 字段的正类比例，`job`（职业）一栏里还可能混着 `unknown` 这种占位值，它既不是空值也不等于"否"，做统计时最容易误导结论。请在下方补全代码，统计 `job` 字段中 `unknown` 占位值的记录数量 `n_unknown_job` 与占比 `share_unknown_job`（占比保留两位小数），同时数出 `y == "yes"` 的认购记录数 `n_subscribed`，并跑通自检。提示：`raw["job"] == "unknown"`、`len(raw)`。


In [ ]:
# 请在下方填写代码
# 目标：统计 job 字段中 unknown 占位值的数量与占比，并统计认购记录数

n_unknown_job = None  # TODO: (raw["job"] == "unknown").sum()
share_unknown_job = None  # TODO: n_unknown_job / len(raw)
n_subscribed = None  # TODO: (raw["y"] == "yes").sum()


In [ ]:
n_unknown_job = (raw["job"] == "unknown").sum()
share_unknown_job = n_unknown_job / len(raw)
n_subscribed = (raw["y"] == "yes").sum()


## 119.10 清理重复并探索响应差异

unknown保留为显式类别，因为未知并不等于否；描述性组间差异不代表营销因果效果。


In [ ]:
df = raw.drop_duplicates().copy()
df["target"] = (df.y == "yes").astype(int)
job_response = (
    df.groupby("job")
    .target.agg(["size", "mean"])
    .query("size>=200")
    .sort_values("mean", ascending=False)
)
contact_response = (
    df.groupby("contact")
    .target.agg(["size", "mean"])
    .sort_values("mean", ascending=False)
)
print("清理后:", len(df), "正类率:", f"{df.target.mean():.2%}")
display(job_response.round(3))
display(contact_response.round(3))


## 119.11 定义预测时点并检查泄漏

duration只有通话结束后才能获得，在通话前响应预测中属于典型事后泄漏。


In [ ]:
forbidden = ["y", "target", "duration"]
features = [column for column in df.columns if column not in forbidden]
_check_1 = bool(not set(features) & set(forbidden))
print(
    "自检 1：not set(features)&set(forbidden) ->",
    "通过" if _check_1 else "需要检查",
)
if not _check_1:
    print("建议：", "请回看输入、处理步骤和预期结果。")
print("预测时点: 通话开始前")
print("禁止字段:", forbidden)
print("可用特征数量:", len(features))
print(
    "duration与目标的组间均值仅用于说明泄漏风险:\n",
    df.groupby("target").duration.mean().round(1),
)


## 119.12 分层划分与Dummy基线

分层划分保持三组正类比例一致；验证集选模型，测试集只用于最终评价。


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score

X = df[features]
y = df.target
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=108
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.2, stratify=y_dev, random_state=108
)
dummy = DummyClassifier(strategy="prior").fit(X_train, y_train)
dummy_probability = dummy.predict_proba(X_val)[:, 1]
print("训练/验证/测试:", len(X_train), len(X_val), len(X_test))
print("正类率:", *[round(part.mean(), 3) for part in [y_train, y_val, y_test]])
print(
    "Dummy PR-AUC:",
    round(average_precision_score(y_val, dummy_probability), 3),
)


## 119.13 建立混合类型预处理Pipeline

类别变量独热编码、数值变量标准化，并把预处理与模型绑定，避免数据处理泄漏。


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

cat = X_train.select_dtypes("object").columns.tolist()
num = [column for column in features if column not in cat]
preprocess = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
        ("num", StandardScaler(), num),
    ]
)
print("类别特征:", len(cat), "数值特征:", len(num))
print("类别示例:", cat[:6])
print("数值示例:", num[:6])


## 119.14 比较逻辑回归与随机森林

两个模型使用相同数据和预处理，在验证集PR-AUC上进行公平比较。


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "逻辑回归": Pipeline(
        [
            ("prep", preprocess),
            (
                "model",
                LogisticRegression(max_iter=700, class_weight="balanced"),
            ),
        ]
    ),
    "随机森林": Pipeline(
        [
            ("prep", preprocess),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=180,
                    min_samples_leaf=10,
                    class_weight="balanced",
                    n_jobs=-1,
                    random_state=108,
                ),
            ),
        ]
    ),
}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    rows.append(
        [
            name,
            average_precision_score(y_val, model.predict_proba(X_val)[:, 1]),
        ]
    )
validation = pd.DataFrame(
    rows, columns=["model", "validation_PR_AUC"]
).sort_values("validation_PR_AUC", ascending=False)
display(validation.round(3))
best_name = validation.iloc[0].model
best_model = models[best_name].fit(X_dev, y_dev)
probability = best_model.predict_proba(X_test)[:, 1]


## 119.15 测试集概率指标评价

类别不平衡时以PR-AUC为主，同时报告ROC-AUC和概率损失。


In [ ]:
from sklearn.metrics import roc_auc_score, log_loss

metrics = pd.Series(
    {
        "ROC_AUC": roc_auc_score(y_test, probability),
        "PR_AUC": average_precision_score(y_test, probability),
        "LogLoss": log_loss(y_test, probability),
        "正类率": y_test.mean(),
    }
)
print("最佳模型:", best_name)
print(metrics.round(3).to_string())


## 119.16 比较Top-K阈值、Lift与覆盖率

Top-K用于教学性阈值评价，展示精确率、召回率和Lift之间的权衡。


In [ ]:
from sklearn.metrics import confusion_matrix

ranked = pd.DataFrame(
    {
        "row_id": X_test.index,
        "actual": y_test.to_numpy(),
        "probability": probability,
    }
).sort_values("probability", ascending=False)
threshold_rows = []
for share in [0.05, 0.10, 0.20]:
    n = max(1, int(len(ranked) * share))
    top = ranked.head(n)
    threshold_rows.append(
        [
            f"{share:.0%}",
            top.probability.min(),
            top.actual.mean(),
            top.actual.sum() / ranked.actual.sum(),
            top.actual.mean() / ranked.actual.mean(),
        ]
    )
threshold_table = pd.DataFrame(
    threshold_rows,
    columns=["Top比例", "概率阈值", "Precision", "Recall", "Lift"],
)
display(threshold_table.round(3))
threshold = threshold_table.loc[
    threshold_table["Top比例"] == "10%", "概率阈值"
].iloc[0]
prediction = probability >= threshold
print("Top10%混淆矩阵:", confusion_matrix(y_test, prediction).tolist())


## 119.17 错误类型与客户分组

区分漏判响应与误报响应，并比较年龄段中的实际率、平均评分和错误率。


In [ ]:
error_df = X_test[["age", "job", "contact", "campaign"]].copy()
error_df["actual"] = y_test
error_df["probability"] = probability
error_df["prediction"] = prediction
error_df["error_type"] = np.select(
    [
        (error_df.actual == 1) & (~error_df.prediction),
        (error_df.actual == 0) & error_df.prediction,
    ],
    ["漏判响应", "误报响应"],
    default="判断正确",
)
error_df["age_group"] = pd.cut(
    error_df.age,
    [0, 30, 45, 60, 120],
    labels=["<=30", "31-45", "46-60", "60+"],
)
age_report = error_df.groupby("age_group", observed=True).agg(
    customers=("actual", "size"),
    actual_rate=("actual", "mean"),
    mean_score=("probability", "mean"),
    error_rate=("error_type", lambda x: (x != "判断正确").mean()),
)
print(error_df.error_type.value_counts())
display(age_report.round(3))


## 119.18 特征解释与模型局限

置换重要性说明模型主要利用哪些历史信号，同时强调响应预测不等于干预效果预测。


In [ ]:
from sklearn.inspection import permutation_importance

sample_n = min(3000, len(X_test))
sample_idx = np.linspace(0, len(X_test) - 1, sample_n, dtype=int)
permutation = permutation_importance(
    best_model,
    X_test.iloc[sample_idx],
    y_test.iloc[sample_idx],
    n_repeats=3,
    scoring="average_precision",
    random_state=108,
    n_jobs=-1,
)
importance = pd.Series(
    permutation.importances_mean, index=features
).sort_values(ascending=False)
print("置换重要性前10:\n", importance.head(10).round(4))
print(
    "局限: 数据来自历史营销活动，unknown较多且存在选择机制；响应概率不能解释电话带来的个体因果增量。"
)


## 119.19 本章实训：从原始记录到质量报告

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C02", "C03", "C04"],
        "amount": [120, 80, 80, None, -20],
    }
)
quality = pd.Series(
    {
        "原始行数": len(raw),
        "重复行数": raw.duplicated().sum(),
        "缺失金额": raw["amount"].isna().sum(),
        "非正金额": (raw["amount"] <= 0).sum(),
    }
)
print(quality.to_string())


### 119.19.1 第一个结果怎么读

项目的第一步不是急着画图或建模，而是量化问题规模。质量报告要能回答：问题有多少、影响哪些字段、下一步如何处理。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
clean = raw.drop_duplicates().copy()
clean["amount_valid"] = clean["amount"].where(clean["amount"] > 0)
summary = clean.groupby("customer_id", as_index=False)["amount_valid"].sum(
    min_count=1
)
print("清洗后行数：", len(clean))
print("有效客户数：", summary["amount_valid"].notna().sum())
print(summary)


### 119.19.2 第二个结果怎么读

第二个实验把质量问题转成可追踪的清洗结果。请同时记录删除、保留和缺失处理规则，不能只报告最后的数字。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 119.20 错误恢复：重复主键和缺失值怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

raw = pd.DataFrame(
    {
        "customer_id": ["C01", "C02", "C02", "C03"],
        "amount": [120, 80, None, -20],
    }
)
duplicate_keys = raw["customer_id"].duplicated(keep=False)
invalid_amount = raw["amount"].isna() | raw["amount"].le(0)
print("重复主键行：")
print(raw[duplicate_keys])
print("金额异常行：")
print(raw[invalid_amount])
print("先标记问题，再决定保留、合并或回查。")


### 119.20.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

项目中不能把异常行静默删除。先输出问题记录和数量，再把处理规则写进项目结论。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 119.21 易错点提醒

**易错点 1**：duration（通话时长）是事后才有的变量，用它做特征会让模型"预知"结果；预测特征必须限制在通话开始之前可获取的信息。

**易错点 2**：正类占比很低，准确率没有参考价值；用召回率/精确率/提升度评价，并与"全都不营销"基线对比。

**易错点 3**：同一客户多次接触（previous 等字段）在训练与测试之间可能重叠，按客户分组切分避免同客户泄漏。

**易错点 4**：job、education 等分类列的缺失/unknown 是真实业务状态，不要粗暴删除，单独成类或保留。

**易错点 5**：month/day_of_week 是类别列，转数值时不要直接编码成 1-12 的连续值（12 月与 1 月不连续），用哑变量或周期编码。


## 119.22 结论与表达

- 预测时点决定duration为什么必须排除
- 分层划分保证类别不平衡下的可比性
- PR-AUC和Top-K指标比准确率更有信息
- 响应预测模型不等于因果增量模型


## 119.23 项目验收清单

- 完成重复、unknown和类别比例审计
- 排除duration及目标字段
- 完成分层三级划分和Dummy基线
- 比较两个Pipeline模型
- 报告PR-AUC、Top-K、Lift与混淆矩阵
- 完成错误分组与置换重要性分析

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 119.24 小结

使用 UCI Bank Marketing 公开数据，按照二分类教学流程建立客户响应预测模型，重点学习事后泄漏、类别不平衡和阈值评价。


### 119.24.1 你已经完成

- 审计重复、unknown和正类比例
- 识别并排除duration事后泄漏
- 使用分层训练验证测试划分
- 建立混合类型预处理Pipeline
- 比较Dummy、逻辑回归和随机森林
- 使用PR-AUC、Top-K、错误切片和特征重要性理解模型


### 119.24.2 质量与结论提醒

- 分号分隔及字段类型
- unknown的数量和含义
- 重复记录与campaign长尾
- 预测时点决定duration为什么必须排除
- 分层划分保证类别不平衡下的可比性
- PR-AUC和Top-K指标比准确率更有信息
- 响应预测模型不等于因果增量模型


### 119.24.3 学习检查

- [ ] 完成重复、unknown和类别比例审计
- [ ] 排除duration及目标字段
- [ ] 完成分层三级划分和Dummy基线
- [ ] 比较两个Pipeline模型
- [ ] 报告PR-AUC、Top-K、Lift与混淆矩阵
- [ ] 完成错误分组与置换重要性分析


### 119.24.4 后续迭代建议

完成验收后，记录一个最值得继续验证的假设：可以是更多数据、不同时间窗口、另一种模型，或一个更细的分组分析。
